# Optimización de Modelos Conjunto Soleado por GMM

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('/content/drive/MyDrive/Tesina/03_Clusterizacion.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,7,77,0,4,15,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,7,82,0,4,16,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,9,85,0,3,16,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,11,87,0,3,16,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,11,88,0,3,16,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,15,86,0,3,14,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,47,89,0,3,16,6,Nublado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,51,95,0,4,17,7,Nublado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,47,100,0,3,18,8,Nublado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,5,100,1,4,18,9,Nublado,Lluvioso


In [3]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [4]:
datos_dia = datos[datos["Cluster GMM"] == "Lluvioso"].copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
30,2022-09-02 06:00:00,0.000000,17,7,91,0,4,15,6,Nublado,Lluvioso,0.000000,0.000000
31,2022-09-02 07:00:00,0.000000,17,7,94,0,3,16,7,Nublado,Lluvioso,0.000000,6.584959
32,2022-09-02 08:00:00,438.814997,16,5,97,0,3,15,8,Nublado,Lluvioso,0.000000,560.422022
33,2022-09-02 09:00:00,5908.000884,17,0,93,1,2,16,9,Nublado,Lluvioso,438.814997,7720.582326
34,2022-09-02 10:00:00,5030.740421,18,0,85,2,2,15,10,Nublado,Lluvioso,5908.000884,9433.109309
38,2022-09-02 14:00:00,28500.000000,24,3,37,6,2,9,14,Soleado,Lluvioso,20596.278869,29057.585772
39,2022-09-02 15:00:00,24647.568577,26,7,33,5,4,8,15,Nublado,Lluvioso,28500.000000,30000.000000
40,2022-09-02 16:00:00,25500.000000,27,7,34,4,4,9,16,Nublado,Lluvioso,24647.568577,28062.328964
41,2022-09-02 17:00:00,24281.956494,28,7,36,2,4,12,17,Nublado,Lluvioso,25500.000000,28786.629243
42,2022-09-02 18:00:00,22733.515002,26,7,39,1,4,12,18,Nublado,Lluvioso,24281.956494,29900.303971


In [5]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [6]:
X = datos_dia[columns]
X

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
30,17,7,91,0,4,15,6,0.000000,0.000000
31,17,7,94,0,3,16,7,0.000000,6.584959
32,16,5,97,0,3,15,8,0.000000,560.422022
33,17,0,93,1,2,16,9,438.814997,7720.582326
34,18,0,85,2,2,15,10,5908.000884,9433.109309
...,...,...,...,...,...,...,...,...,...
18273,14,0,87,1,4,11,8,67.000000,7302.000000
18274,15,0,83,2,4,12,9,7356.000000,18014.000000
18275,17,0,71,4,3,12,10,17638.000000,23010.000000
18276,19,0,60,5,3,11,11,23339.000000,26156.000000


In [7]:
y = datos_dia[['Generación']]
y

,Generación
30,0.000000
31,0.000000
32,438.814997
33,5908.000884
34,5030.740421
...,...
18273,7356.000000
18274,17638.000000
18275,23339.000000
18276,26323.000000


Dividimos entrenamiento, validación y prueba

In [8]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [9]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 4263, y_train: 4263
X_val: 913, y_val: 913
X_test: 914, y_test: 914


## Escalar con MinMaxScaler

In [10]:
from sklearn.preprocessing import MinMaxScaler

In [11]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [12]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[4.70588235e-01 7.77777778e-02 9.04255319e-01 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [4.70588235e-01 7.77777778e-02 9.36170213e-01 ... 6.66666667e-02
  0.00000000e+00 2.19498623e-04]
 [4.41176471e-01 5.55555556e-02 9.68085106e-01 ... 1.33333333e-01
  0.00000000e+00 1.86807341e-02]
 ...
 [7.35294118e-01 0.00000000e+00 2.65957447e-01 ... 7.33333333e-01
  8.68733333e-01 6.41966667e-01]
 [6.47058824e-01 0.00000000e+00 3.93617021e-01 ... 8.00000000e-01
  8.07700000e-01 4.83833333e-01]
 [5.88235294e-01 2.22222222e-02 5.21276596e-01 ... 8.66666667e-01
  5.96633333e-01 9.58000000e-02]]
(4263, 9)


In [13]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
30,0.470588,0.077778,0.904255,0.000000,0.75,0.789474,0.000000,0.000000,0.000000
31,0.470588,0.077778,0.936170,0.000000,0.50,0.842105,0.066667,0.000000,0.000219
32,0.441176,0.055556,0.968085,0.000000,0.50,0.789474,0.133333,0.000000,0.018681
33,0.470588,0.000000,0.925532,0.071429,0.25,0.842105,0.200000,0.014627,0.257353
34,0.500000,0.000000,0.840426,0.142857,0.25,0.789474,0.266667,0.196933,0.314437
...,...,...,...,...,...,...,...,...,...
13522,0.382353,0.000000,0.776596,0.142857,0.50,0.578947,0.200000,0.415467,0.612933
13529,0.794118,0.000000,0.202128,0.142857,1.00,0.315789,0.666667,0.862067,0.847600
13530,0.735294,0.000000,0.265957,0.071429,1.00,0.368421,0.733333,0.868733,0.641967
13531,0.647059,0.000000,0.393617,0.071429,1.00,0.526316,0.800000,0.807700,0.483833


In [14]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.55882353 0.48888889 0.61702128 ... 0.93333333 0.11096667 0.        ]
 [0.5        0.56666667 0.72340426 ... 1.         0.         0.        ]
 [0.38235294 0.22222222 0.82978723 ... 0.         0.         0.        ]
 ...
 [0.58823529 0.56666667 0.65957447 ... 0.33333333 0.89683333 0.92476667]
 [0.64705882 0.52222222 0.57446809 ... 0.4        0.92476667 0.93596667]
 [0.67647059 0.44444444 0.5106383  ... 0.46666667 0.9375     0.9438    ]]
(913, 9)


In [15]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
13533,0.558824,0.488889,0.617021,0.000000,1.0,0.684211,0.933333,0.110967,0.000000
13534,0.500000,0.566667,0.723404,0.000000,1.0,0.736842,1.000000,0.000000,0.000000
13543,0.382353,0.222222,0.829787,0.000000,1.0,0.578947,0.000000,0.000000,0.000000
13544,0.382353,0.177778,0.840426,0.000000,1.0,0.578947,0.066667,0.000000,0.000300
13545,0.352941,0.077778,0.851064,0.071429,1.0,0.578947,0.133333,0.000067,0.415467
...,...,...,...,...,...,...,...,...,...
16546,0.500000,0.411111,0.819149,0.214286,0.5,0.789474,0.200000,0.435333,0.795400
16547,0.529412,0.444444,0.744681,0.285714,1.0,0.789474,0.266667,0.795400,0.900500
16548,0.588235,0.566667,0.659574,0.571429,1.0,0.789474,0.333333,0.896833,0.924767
16549,0.647059,0.522222,0.574468,0.857143,0.5,0.789474,0.400000,0.924767,0.935967


In [16]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.73529412 0.35555556 0.46808511 ... 0.53333333 0.95086667 0.95016667]
 [0.73529412 0.35555556 0.44680851 ... 0.6        0.95153333 0.95743333]
 [0.76470588 0.4        0.42553191 ... 0.66666667 0.95743333 0.8989    ]
 ...
 [0.47058824 0.         0.69148936 ... 0.26666667 0.58793333 0.767     ]
 [0.52941176 0.         0.57446809 ... 0.33333333 0.77796667 0.87186667]
 [0.64705882 0.         0.40425532 ... 0.46666667 0.8759     0.8551    ]]
(914, 9)


In [17]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
16551,0.735294,0.355556,0.468085,0.857143,0.25,0.736842,0.533333,0.950867,0.950167
16552,0.735294,0.355556,0.446809,0.642857,0.25,0.789474,0.600000,0.951533,0.957433
16553,0.764706,0.400000,0.425532,0.357143,0.00,0.789474,0.666667,0.957433,0.898900
16554,0.735294,0.488889,0.446809,0.214286,0.25,0.789474,0.733333,0.898900,0.878400
16555,0.705882,0.488889,0.500000,0.071429,1.00,0.789474,0.800000,0.877867,0.777867
...,...,...,...,...,...,...,...,...,...
18273,0.382353,0.000000,0.861702,0.071429,0.75,0.578947,0.133333,0.002233,0.243400
18274,0.411765,0.000000,0.819149,0.142857,0.75,0.631579,0.200000,0.245200,0.600467
18275,0.470588,0.000000,0.691489,0.285714,0.50,0.631579,0.266667,0.587933,0.767000
18276,0.529412,0.000000,0.574468,0.357143,0.50,0.578947,0.333333,0.777967,0.871867


In [18]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [19]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[4.32432432e-01 7.77777778e-02 9.06250000e-01 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [4.32432432e-01 7.77777778e-02 9.37500000e-01 ... 6.66666667e-02
  0.00000000e+00 2.19498623e-04]
 [4.05405405e-01 5.55555556e-02 9.68750000e-01 ... 1.33333333e-01
  0.00000000e+00 1.86807341e-02]
 ...
 [4.32432432e-01 0.00000000e+00 6.97916667e-01 ... 2.66666667e-01
  5.87933333e-01 7.67000000e-01]
 [4.86486486e-01 0.00000000e+00 5.83333333e-01 ... 3.33333333e-01
  7.77966667e-01 8.71866667e-01]
 [5.94594595e-01 0.00000000e+00 4.16666667e-01 ... 4.66666667e-01
  8.75900000e-01 8.55100000e-01]]
(6090, 9)


In [20]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
30,0.432432,0.077778,0.906250,0.000000,0.75,0.75,0.000000,0.000000,0.000000
31,0.432432,0.077778,0.937500,0.000000,0.50,0.80,0.066667,0.000000,0.000219
32,0.405405,0.055556,0.968750,0.000000,0.50,0.75,0.133333,0.000000,0.018681
33,0.432432,0.000000,0.927083,0.071429,0.25,0.80,0.200000,0.014627,0.257353
34,0.459459,0.000000,0.843750,0.142857,0.25,0.75,0.266667,0.196933,0.314437
...,...,...,...,...,...,...,...,...,...
18273,0.351351,0.000000,0.864583,0.071429,0.75,0.55,0.133333,0.002233,0.243400
18274,0.378378,0.000000,0.822917,0.142857,0.75,0.60,0.200000,0.245200,0.600467
18275,0.432432,0.000000,0.697917,0.285714,0.50,0.60,0.266667,0.587933,0.767000
18276,0.486486,0.000000,0.583333,0.357143,0.50,0.55,0.333333,0.777967,0.871867


In [21]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [22]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.        ]
 [0.        ]
 [0.01462717]
 ...
 [0.8077    ]
 [0.59663333]
 [0.11096667]]
(4263, 1)


In [23]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
30,0.000000
31,0.000000
32,0.014627
33,0.196933
34,0.167691
...,...
13522,0.444233
13529,0.868733
13530,0.807700
13531,0.596633


In [24]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [6.66666667e-05]
 [1.89933333e-01]
 [6.10200000e-01]
 [7.91700000e-01]
 [8.05833333e-01]
 [7.12233333e-01]
 [6.85733333e-01]
 [6.81133333e-01]
 [7.68600000e-01]
 [9.80400000e-01]
 [7.70233333e-01]
 [4.56866667e-01]
 [1.07900000e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [3.56133333e-01]
 [4.50066667e-01]
 [6.51266667e-01]
 [9.75433333e-01]
 [9.28733333e-01]
 [9.71900000e-01]
 [9.31933333e-01]
 [5.67633333e-01]
 [6.32233333e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.00000000e-04]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [4.74833333e-01]
 [6.28000000e-01]
 [7.55433333e-01]
 [7.36500000e-01]
 [7.29200000e-01]
 [6.37000000e-01]
 [7.18033333e-01]
 [5.17733333e-01]
 [1.03866667e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.66666667e-04]
 [0.00000000e+00]
 [6.66666667e-05]
 [5.93533333e-01]
 [6.18800000e-01]
 [8.85000000e-01]
 [8.953666

In [25]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
13533,0.000000
13534,0.000000
13543,0.000000
13544,0.000067
13545,0.189933
...,...
16546,0.795400
16547,0.896833
16548,0.924767
16549,0.937500


In [26]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[9.51533333e-01]
 [9.57433333e-01]
 [8.98900000e-01]
 [8.77866667e-01]
 [7.85400000e-01]
 [3.59033333e-01]
 [3.83666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [4.49000000e-02]
 [4.35333333e-01]
 [7.95400000e-01]
 [9.33400000e-01]
 [8.52400000e-01]
 [8.32100000e-01]
 [9.52200000e-01]
 [6.61300000e-01]
 [6.58233333e-01]
 [5.76366667e-01]
 [4.95800000e-01]
 [3.77400000e-01]
 [1.65566667e-01]
 [1.78666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.93000000e-02]
 [1.40800000e-01]
 [2.47800000e-01]
 [3.81700000e-01]
 [3.92633333e-01]
 [3.94533333e-01]
 [4.66566667e-01]
 [4.48600000e-01]
 [5.12533333e-01]
 [5.03500000e-01]
 [6.21466667e-01]
 [7.75366667e-01]
 [1.92633333e-01]
 [1.78666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [4.38333333e-02]
 [4.69166667e-01]
 [8.33500000e-01]
 [8.16000000e-01]
 [6.67466667e-01]
 [6.60733333e-01]
 [6.64166667e-01]
 [6.88000000e-01]
 [6.62833333e-01]
 [6.58766667e-01]
 [6.36333333e-01]
 [8.15333333e-01]
 [4.41033333e-01]
 [4.64000000e-02]
 [0.000000

In [27]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
16551,0.951533
16552,0.957433
16553,0.898900
16554,0.877867
16555,0.785400
...,...
18273,0.245200
18274,0.587933
18275,0.777967
18276,0.877433


In [28]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [29]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.        ]
 [0.        ]
 [0.01462717]
 ...
 [0.77796667]
 [0.87743333]
 [0.85326667]]
(6090, 1)


In [30]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
30,0.000000
31,0.000000
32,0.014627
33,0.196933
34,0.167691
...,...
18273,0.245200
18274,0.587933
18275,0.777967
18276,0.877433


## Preparación para Redes Neuronales

In [31]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []

    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada

        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [32]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [33]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (4215, 48, 9), y_train: (4215, 1)
X_val: (865, 48, 9), y_val: (865, 1)
X_test: (866, 48, 9), y_test: (866, 1)


## Optuna

In [34]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.8/231.8 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 9.7 MB/s eta 0:00:00


In [35]:
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

### Optimización de LightGBM

In [36]:
import optuna
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

# Definir la función de optimización
def objective(trial):
    # Espacio de búsqueda para los hiperparámetros
    num_leaves = trial.suggest_int('num_leaves', 10, 1000)
    subsample = trial.suggest_float('subsample', 0.1, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.1, 1.0)
    min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 10, 100)

    # Modelo LightGBM con los hiperparámetros sugeridos
    model = LGBMRegressor(
        num_leaves=num_leaves,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_data_in_leaf=min_data_in_leaf,
        random_state=42
    )

    # Entrenamos el modelo con los datos de entrenamiento
    model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Ejecutar la optimización con 50 iteraciones
LightGBM_study = optuna.create_study(direction="minimize")
LightGBM_study.optimize(objective, n_trials=50)

# Obtener los mejores hiperparámetros
best_params = LightGBM_study.best_params
print("Mejores hiperparámetros:", best_params)

[I 2025-03-13 21:17:28,688] A new study created in memory with name: no-name-b6484cb0-e1fb-4a8f-9346-56106e13ac34


[LightGBM] [Warning] min_data_in_leaf is set=21, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=21, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=21
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001323 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 756
[LightGBM] [Info] Number of data points in the train set: 4263, number of used features: 9
[LightGBM] [Info] Start training from score 0.312844
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posi

[I 2025-03-13 21:17:29,039] Trial 0 finished with value: 0.010606304057121256 and parameters: {'num_leaves': 716, 'subsample': 0.1609883544172853, 'colsample_bytree': 0.8565395529562011, 'min_data_in_leaf': 21}. Best is trial 0 with value: 0.010606304057121256.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-13 21:17:29,168] Trial 1 finished with value: 0.009870112928837516 and parameters: {'num_leaves': 389, 'subsample': 0.9004551205794232, 'colsample_bytree': 0.9352648739342498, 'min_data_in_leaf': 63}. Best is trial 1 with value: 0.009870112928837516.


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-13 21:17:34,688] Trial 48 finished with value: 0.009445032194761583 and parameters: {'num_leaves': 795, 'subsample': 0.5076726972898223, 'colsample_bytree': 0.3581049003597675, 'min_data_in_leaf': 84}. Best is trial 27 with value: 0.00883967604525281.


[LightGBM] [Warning] min_data_in_leaf is set=84, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=84
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=84, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=84
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000172 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 756
[LightGBM] [Info] Number of data points in the train set: 4263, number of used features: 9
[LightGBM] [Info] Start training from score 0.312844
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

[I 2025-03-13 21:17:34,822] Trial 49 finished with value: 0.009759269123743785 and parameters: {'num_leaves': 58, 'subsample': 0.2986072317768991, 'colsample_bytree': 0.8407144960562211, 'min_data_in_leaf': 37}. Best is trial 27 with value: 0.00883967604525281.


[LightGBM] [Warning] min_data_in_leaf is set=37, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=37
Mejores hiperparámetros: {'num_leaves': 112, 'subsample': 0.38768735737758864, 'colsample_bytree': 0.6849851431944389, 'min_data_in_leaf': 94}


### Random Forest

In [37]:
def objective(trial):
    # Definir los hiperparámetros a optimizar
    n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)
    max_depth = trial.suggest_int("max_depth", 10, 50, step=5)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    # Definir el modelo con los hiperparámetros actuales
    RF_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        criterion="squared_error",
        random_state=0
    )

    # Entrenar el modelo
    RF_model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = RF_model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Crear el estudio de optimización
RF_study = optuna.create_study(direction="minimize")
RF_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", RF_study.best_params)

[I 2025-03-13 21:17:34,831] A new study created in memory with name: no-name-7fc24bf8-cf75-4cf4-b611-3d736f78090e
[I 2025-03-13 21:17:41,550] Trial 0 finished with value: 0.016798185365360693 and parameters: {'n_estimators': 400, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 2, 'bootstrap': False}. Best is trial 0 with value: 0.016798185365360693.
[I 2025-03-13 21:17:44,421] Trial 1 finished with value: 0.01037393477622574 and parameters: {'n_estimators': 200, 'max_depth': 15, 'min_samples_split': 3, 'min_samples_leaf': 3, 'bootstrap': True}. Best is trial 1 with value: 0.01037393477622574.
[I 2025-03-13 21:17:46,102] Trial 2 finished with value: 0.010273227871832323 and parameters: {'n_estimators': 150, 'max_depth': 45, 'min_samples_split': 15, 'min_samples_leaf': 10, 'bootstrap': True}. Best is trial 2 with value: 0.010273227871832323.
[I 2025-03-13 21:17:50,941] Trial 3 finished with value: 0.02061744443223697 and parameters: {'n_estimators': 200, 'max_depth': 50, 'mi

Mejores hiperparámetros: {'n_estimators': 250, 'max_depth': 10, 'min_samples_split': 8, 'min_samples_leaf': 6, 'bootstrap': True}


### CTNET

In [38]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])

    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

# Construir modelo con hiperparámetros sugeridos
def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout, mlp_dropout):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs

    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)

    for units in mlp_units:
        x = layers.Dense(units, activation="relu")(x)

    x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(1)(x)
    return tf.keras.Model(inputs, outputs)

In [39]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [40]:
# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    head_size = trial.suggest_int("head_size", 2, 8)
    num_heads = trial.suggest_int("num_heads", 2, 8)
    ff_dim = trial.suggest_int("ff_dim", 16, 128, step=16)
    num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 5)
    mlp_units = [trial.suggest_int("mlp_units_1", 64, 512, step=64),
                 trial.suggest_int("mlp_units_2", 32, 256, step=32)]
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Construir el modelo
    model = build_model(
        input_shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]),
        head_size=head_size,
        num_heads=num_heads,
        ff_dim=ff_dim,
        num_transformer_blocks=num_transformer_blocks,
        mlp_units=mlp_units,
        dropout=dropout,
        mlp_dropout=mlp_dropout
    )

    # Compilar y entrenar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Minimizar MSE

# Crear el estudio de optimización
CTNET_study = optuna.create_study(direction="minimize")
CTNET_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", CTNET_study.best_params)


[I 2025-03-13 21:20:35,657] A new study created in memory with name: no-name-5902e74d-938c-46ff-8626-e70e1c000cdb
[I 2025-03-13 21:21:41,823] Trial 0 finished with value: 0.06313987821340561 and parameters: {'head_size': 2, 'num_heads': 8, 'ff_dim': 128, 'num_transformer_blocks': 2, 'mlp_units_1': 448, 'mlp_units_2': 160, 'dropout': 0.4413695203721937, 'mlp_dropout': 0.24394014858288582, 'learning_rate': 0.0001434197963790112, 'batch_size': 512}. Best is trial 0 with value: 0.06313987821340561.
[I 2025-03-13 21:22:44,076] Trial 1 finished with value: 0.11984812468290329 and parameters: {'head_size': 7, 'num_heads': 3, 'ff_dim': 96, 'num_transformer_blocks': 3, 'mlp_units_1': 64, 'mlp_units_2': 224, 'dropout': 0.1511918435268007, 'mlp_dropout': 0.10968943678189724, 'learning_rate': 5.885403729299824e-05, 'batch_size': 512}. Best is trial 0 with value: 0.06313987821340561.
[I 2025-03-13 21:24:12,853] Trial 2 finished with value: 0.059544216841459274 and parameters: {'head_size': 3, 'num_

Mejores hiperparámetros: {'head_size': 7, 'num_heads': 2, 'ff_dim': 96, 'num_transformer_blocks': 4, 'mlp_units_1': 128, 'mlp_units_2': 224, 'dropout': 0.1267581143378228, 'mlp_dropout': 0.313337321637589, 'learning_rate': 0.0008212627019643434, 'batch_size': 256}


### Forescasting

In [42]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Conv1D, BatchNormalization, MaxPooling1D, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters = trial.suggest_categorical("filters", [32, 64, 128])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    lstm_units_1 = trial.suggest_categorical("lstm_units_1", [64, 128, 256])
    lstm_units_2 = trial.suggest_categorical("lstm_units_2", [32, 64, 128])
    lstm_units_3 = trial.suggest_categorical("lstm_units_3", [16, 32, 64])
    dropout_lstm = trial.suggest_float("dropout_lstm", 0.1, 0.5)
    dropout_dense = trial.suggest_float("dropout_dense", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])  # 🔹 Optimización del batch size

    # Construcción del modelo
    model = Sequential()
    model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

    # CNN
    model.add(Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))

    # BiLSTM
    model.add(Bidirectional(LSTM(lstm_units_1, return_sequences=True)))
    model.add(Bidirectional(LSTM(lstm_units_2, return_sequences=True)))
    model.add(Dropout(dropout_lstm))
    model.add(Bidirectional(LSTM(lstm_units_3, return_sequences=False)))

    # Normalización y Dropout
    model.add(BatchNormalization())
    model.add(Dropout(dropout_dense))

    # Capas Densas
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='relu'))

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Entrenar el modelo con validación
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Forecasting_study = optuna.create_study(direction="minimize")
Forecasting_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Forecasting_study.best_params)

[I 2025-03-13 22:16:46,671] A new study created in memory with name: no-name-ce40fb51-92f9-4466-bb56-bd17f1ca9330
[I 2025-03-13 22:18:01,263] Trial 8 finished with value: 0.30854707956314087 and parameters: {'filters': 32, 'kernel_size': 4, 'lstm_units_1': 256, 'lstm_units_2': 64, 'lstm_units_3': 16, 'dropout_lstm': 0.2686336137690571, 'dropout_dense': 0.2084684018020383, 'learning_rate': 0.0007993538793772589, 'batch_size': 256}. Best is trial 8 with value: 0.30854707956314087.
[I 2025-03-13 22:18:12,021] Trial 12 finished with value: 0.2977001965045929 and parameters: {'filters': 128, 'kernel_size': 5, 'lstm_units_1': 128, 'lstm_units_2': 64, 'lstm_units_3': 32, 'dropout_lstm': 0.22627100465176167, 'dropout_dense': 0.4283638114552596, 'learning_rate': 0.00037399963913365493, 'batch_size': 128}. Best is trial 12 with value: 0.2977001965045929.
[I 2025-03-13 22:18:23,725] Trial 13 finished with value: 0.18192628026008606 and parameters: {'filters': 32, 'kernel_size': 3, 'lstm_units_1':

Mejores hiperparámetros: {'filters': 64, 'kernel_size': 4, 'lstm_units_1': 256, 'lstm_units_2': 32, 'lstm_units_3': 64, 'dropout_lstm': 0.25096267684751133, 'dropout_dense': 0.22780158875618112, 'learning_rate': 0.009446162950489861, 'batch_size': 256}


### Photovoltaic

In [43]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Bidirectional, GRU, MultiHeadAttention
from tensorflow.keras.layers import Flatten, Dropout, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters_1 = trial.suggest_categorical("filters_1", [32, 64, 128])
    filters_2 = trial.suggest_categorical("filters_2", [64, 128, 256])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    gru_units = trial.suggest_categorical("gru_units", [32, 64, 128])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8])
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    dense_units_1 = trial.suggest_categorical("dense_units_1", [32, 64, 128])
    dense_units_2 = trial.suggest_categorical("dense_units_2", [16, 32, 64])
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Definir el modelo
    inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

    # Primera capa CNN
    x = Conv1D(filters=filters_1, kernel_size=kernel_size, padding='same', activation='relu')(inputs)
    x = MaxPooling1D(pool_size=2)(x)

    # Segunda capa CNN
    x = Conv1D(filters=filters_2, kernel_size=kernel_size, padding='same', activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)

    # Capa BiGRU
    x = Bidirectional(GRU(gru_units, return_sequences=True))(x)

    # Atención MultiHead
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=128)(x, x)

    # Aplanar y Dropout
    x = Flatten()(attention)
    x = Dropout(dropout_rate)(x)
    x = Dense(dense_units_1, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = Dense(dense_units_2, activation="relu")(x)

    # Capa de salida
    outputs = Dense(1, activation="linear")(x)

    # Construcción del modelo
    model = Model(inputs=inputs, outputs=outputs)

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Early Stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

    # Entrenar el modelo
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se detendrá con Early Stopping
        batch_size=batch_size,
        validation_data=(X_val_windowed, y_val_windowed),
        callbacks=[early_stop],
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Photovoltaic_study = optuna.create_study(direction="minimize")
Photovoltaic_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Photovoltaic_study.best_params)

[I 2025-03-13 22:29:05,961] A new study created in memory with name: no-name-478fec80-99d7-415d-91a6-fbb1b61c568a


Epoch 16: early stopping
Restoring model weights from the end of the best epoch: 6.


[I 2025-03-13 22:30:23,007] Trial 6 finished with value: 0.08595424890518188 and parameters: {'filters_1': 64, 'filters_2': 64, 'kernel_size': 3, 'gru_units': 128, 'num_heads': 4, 'dropout_rate': 0.37232030188564647, 'dense_units_1': 64, 'dense_units_2': 64, 'learning_rate': 0.003671270823589523, 'batch_size': 256}. Best is trial 6 with value: 0.08595424890518188.


Epoch 40: early stopping
Restoring model weights from the end of the best epoch: 30.


[I 2025-03-13 22:30:32,243] Trial 0 finished with value: 0.04263355955481529 and parameters: {'filters_1': 128, 'filters_2': 128, 'kernel_size': 2, 'gru_units': 64, 'num_heads': 2, 'dropout_rate': 0.24444674258737553, 'dense_units_1': 64, 'dense_units_2': 16, 'learning_rate': 0.0018964692294545682, 'batch_size': 512}. Best is trial 0 with value: 0.04263355955481529.


...

Epoch 35: early stopping
Restoring model weights from the end of the best epoch: 25.


[I 2025-03-13 22:36:32,477] Trial 49 finished with value: 0.045115016400814056 and parameters: {'filters_1': 128, 'filters_2': 256, 'kernel_size': 3, 'gru_units': 128, 'num_heads': 2, 'dropout_rate': 0.27708888859385405, 'dense_units_1': 64, 'dense_units_2': 16, 'learning_rate': 0.0011343799762559424, 'batch_size': 512}. Best is trial 42 with value: 0.04142600670456886.


Epoch 47: early stopping
Restoring model weights from the end of the best epoch: 37.


[I 2025-03-13 22:36:34,610] Trial 48 finished with value: 0.04386664927005768 and parameters: {'filters_1': 128, 'filters_2': 256, 'kernel_size': 3, 'gru_units': 32, 'num_heads': 8, 'dropout_rate': 0.2844190834547703, 'dense_units_1': 64, 'dense_units_2': 16, 'learning_rate': 0.0009913501078083822, 'batch_size': 512}. Best is trial 42 with value: 0.04142600670456886.


Mejores hiperparámetros: {'filters_1': 128, 'filters_2': 256, 'kernel_size': 2, 'gru_units': 32, 'num_heads': 8, 'dropout_rate': 0.3467902144595288, 'dense_units_1': 64, 'dense_units_2': 16, 'learning_rate': 0.002342745284309666, 'batch_size': 512}
